# 第3讲 课程代码案例：Spec-driven 协作建模 × Python 工程化

> 课程：Python 进阶 | 教师：孙青 / 欧阳元新 | 平台：CloudStudio + CodeBuddy
>
> **AI 角色定位 = 受控开发者**：Spec 定义约束 → AI 在边界内实现 → 人审查验收
>
> **贯穿案例**：科研实验数据管理与分析工具（sci_analyzer）

## Part 1：Python 工程化基础

**受众**：已完成前两讲实验、能写数据分析脚本的学生

**前置条件**：有 Python 函数和文件操作基础

**学习目标**：
1. 掌握 Python 包结构（模块化拆分、`__init__.py`、导入机制）
2. 掌握类型注解（typing 模块、函数签名）
3. 掌握异常处理（try/except 层级、自定义异常）
4. 掌握 pytest 单元测试（断言/fixture/参数化）

### 1.1 从单文件到模块化：为什么要拆分

In [1]:
# 演示：一个"能跑但不可维护"的脚本
# 改造前：300 行混在一个文件里
# - 数据加载、清洗、统计、可视化全部混合
# - 想改某一步？先花 10 分钟找到它在哪里
# - 想复用清洗逻辑？只能复制粘贴

# 改造后的结构
good_structure = (
    "sci_analyzer/\n"
    "  __init__.py          # 包入口，导出公开接口\n"
    "  loader.py            # 数据加载（职责单一）\n"
    "  cleaner.py           # 数据清洗（职责单一）\n"
    "  stats_engine.py      # 统计分析（职责单一）\n"
    "  visualizer.py        # 可视化（职责单一）\n"
    "  exceptions.py        # 自定义异常\n"
    "tests/\n"
    "  test_loader.py\n"
    "  test_cleaner.py\n"
    "  conftest.py          # 共享 fixture\n"
    "pyproject.toml           # 项目配置"
)
print(good_structure)
print()
print("原则：一个模块只做一件事（Single Responsibility）")

sci_analyzer/
  __init__.py          # 包入口，导出公开接口
  loader.py            # 数据加载（职责单一）
  cleaner.py           # 数据清洗（职责单一）
  stats_engine.py      # 统计分析（职责单一）
  visualizer.py        # 可视化（职责单一）
  exceptions.py        # 自定义异常
tests/
  test_loader.py
  test_cleaner.py
  conftest.py          # 共享 fixture
pyproject.toml           # 项目配置

原则：一个模块只做一件事（Single Responsibility）


### 1.2 创建包结构：__init__.py

In [2]:
# 创建项目目录
import os
from pathlib import Path

project_dir = Path("sci_analyzer")
project_dir.mkdir(exist_ok=True)

init_content = (
    '"""科研实验数据管理与分析工具"""\n\n'
    'from .loader import load_data\n'
    'from .cleaner import clean_data, remove_duplicates, detect_outliers\n'
    'from .stats_engine import run_statistics\n'
    'from .exceptions import DataError, LoadError, CleanError\n\n'
    '__version__ = "0.1.0"\n'
    '__all__ = [\n'
    '    "load_data", "clean_data", "remove_duplicates",\n'
    '    "detect_outliers", "run_statistics",\n'
    '    "DataError", "LoadError", "CleanError",\n'
    ']\n'
)
(project_dir / "__init__.py").write_text(init_content)

print("__init__.py 作用：")
print("1. 标识这是一个 Python 包")
print("2. 控制外部 import 时能看到什么（__all__）")
print("3. 提供包级别的元数据（__version__）")

__init__.py 作用：
1. 标识这是一个 Python 包
2. 控制外部 import 时能看到什么（__all__）
3. 提供包级别的元数据（__version__）


### 1.3 类型注解：让代码自文档化

In [3]:
from typing import Optional, Any
from pathlib import Path
import pandas as pd

# 无类型注解：函数接口模糊
def process(data, config):
    pass  # 调用者不知道传什么、返回什么

# 有类型注解：一目了然
def load_data(
    file_path: Path,
    sheet_name: Optional[str] = None,
    encoding: str = "utf-8"
) -> pd.DataFrame:
    """加载数据文件。

    Args:
        file_path: 数据文件路径，支持 .csv/.xlsx/.json
        sheet_name: Excel 工作表名（仅 Excel 格式需要）
        encoding: 文件编码（仅 CSV 格式需要）

    Returns:
        加载后的 DataFrame

    Raises:
        FileNotFoundError: 文件不存在
        ValueError: 不支持的文件格式
    """
    pass

# 常用类型注解
from typing import Union

def detect_outliers(
    series: pd.Series,
    method: str = "iqr",
    threshold: float = 1.5
) -> tuple[pd.Series, dict[str, Any]]:
    """返回 (布尔掩码, 统计信息字典)"""
    pass

print("类型注解三大好处：")
print("1. IDE 自动补全更精准")
print("2. AI 生成的代码更容易审查")
print("3. 代码即文档，减少沟通成本")

类型注解三大好处：
1. IDE 自动补全更精准
2. AI 生成的代码更容易审查
3. 代码即文档，减少沟通成本


### 1.4 异常处理：优雅应对错误

In [4]:
# 自定义异常层级
class DataError(Exception):
    """数据处理相关错误的基类"""
    pass

class LoadError(DataError):
    """数据加载失败"""
    pass

class CleanError(DataError):
    """数据清洗失败"""
    def __init__(self, column: str, reason: str):
        self.column = column
        super().__init__(f"列\'{column}\'清洗失败: {reason}")

# 异常处理实战：数据加载
def load_data_demo(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"文件不存在: {file_path}")

    suffix = file_path.suffix.lower()
    if suffix not in ('.csv', '.xlsx', '.xls', '.json'):
        raise ValueError(f"不支持的格式: {suffix}")

    try:
        if suffix == '.csv':
            return pd.read_csv(file_path, encoding='utf-8-sig')
        elif suffix in ('.xlsx', '.xls'):
            return pd.read_excel(file_path)
        else:
            return pd.read_json(file_path)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding='gbk')
    except Exception as e:
        raise LoadError(f"加载 {file_path.name} 失败") from e

# 测试错误处理
try:
    load_data_demo(Path("不存在的文件.csv"))
except FileNotFoundError as e:
    print(f"正确捕获: {e}")

try:
    load_data_demo(Path("test.txt"))
except ValueError as e:
    print(f"正确捕获: {e}")

正确捕获: 文件不存在: 不存在的文件.csv


FileNotFoundError: 文件不存在: test.txt

### 1.5 pytest 单元测试

In [5]:
# 写入测试文件演示
from pathlib import Path
import textwrap

tests_dir = Path("tests")
tests_dir.mkdir(exist_ok=True)
(tests_dir / "__init__.py").touch()

test_code = textwrap.dedent('''\
import pytest
import pandas as pd
import numpy as np
from pathlib import Path

@pytest.fixture
def sample_df():
    return pd.DataFrame({
        "name": ["A", "B", "C", "A", "D"],
        "value": [1.0, 2.0, np.nan, 1.0, 100.0],
        "category": ["x", "y", "x", "x", "y"]
    })

@pytest.fixture
def csv_file(tmp_path):
    f = tmp_path / "test.csv"
    f.write_text("name,value\\nA,1\\nB,2\\nC,3")
    return f

def test_load_csv(csv_file):
    df = pd.read_csv(csv_file)
    assert len(df) == 3
    assert list(df.columns) == ["name", "value"]

def test_remove_duplicates(sample_df):
    result = sample_df.drop_duplicates()
    assert len(result) == 4

def test_load_nonexistent():
    with pytest.raises(Exception):
        pd.read_csv(Path("ghost_file.csv"))

@pytest.mark.parametrize("threshold,expected_outliers", [
    (1.5, 1),
    (3.0, 0),
])
def test_outlier_detection(sample_df, threshold, expected_outliers):
    values = sample_df["value"].dropna()
    Q1, Q3 = values.quantile(0.25), values.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - threshold * IQR, Q3 + threshold * IQR
    outliers = values[(values < lower) | (values > upper)]
    assert len(outliers) == expected_outliers
''')

(tests_dir / "test_demo.py").write_text(test_code)
print("测试文件已生成: tests/test_demo.py")
print()
print("运行方式: pytest tests/ -v")
print("参数化测试: 一组代码覆盖多种输入组合")

测试文件已生成: tests/test_demo.py

运行方式: pytest tests/ -v
参数化测试: 一组代码覆盖多种输入组合


In [7]:
# 运行 pytest
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/test_demo.py", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


/Users/jafekin/miniconda3/bin/python: No module named pytest



### 1.6 AI 协作练习：TDD + AI

**TDD（测试驱动开发）+ AI 的工作流**：

```
步骤1：你写测试 → 定义"什么是正确的"
步骤2：让 AI 实现 → "请让这些测试通过"
步骤3：运行 pytest → 自动裁判
步骤4：失败？告知 AI 哪个测试失败 → AI 修改 → 重新验证
```

**Prompt 示例**：
```
我已经写好了 tests/test_cleaner.py（见 @tests/test_cleaner.py）。
请实现 sci_analyzer/cleaner.py，使所有测试通过。
遵守规则：类型注解完整、不修改原输入、用 logging 不用 print。
```

## Part 2：Spec-driven 开发范式

**受众**：已掌握 Part 1 工程化基础的学生

**前置条件**：Part 1 完成

**学习目标**：
1. 理解 Spec-driven 与 Prompt 的层级区别
2. 掌握 Constitution / Context / Tasks 三文档编写
3. 完成一次完整的 Spec → AI 实现 → 验收 循环

### 2.1 三个层级：从 Prompt 到 Spec

In [8]:
levels = {
    "L1 两段式 Prompt": {
        "场景": "单个函数/快速原型",
        "AI自由度": "高（自由发挥）",
        "示例": "帮我写一个读取 CSV 的函数",
    },
    "L2 Rules 约束": {
        "场景": "代码风格统一",
        "AI自由度": "中（风格受限）",
        "示例": ".codebuddy/rules/python-style.md",
    },
    "L3 Spec-driven": {
        "场景": "多模块工程",
        "AI自由度": "低（行为受限）",
        "示例": "speckit.md 定义接口+约束+验收标准",
    },
}

print("=== AI 协作三层级 ===")
print()
for level, info in levels.items():
    print(f"{level}")
    for k, v in info.items():
        print(f"  {k}: {v}")
    print()

print("进阶路径：第1讲 L1 → 第2讲 L1+L2 → 第3讲 L3")

=== AI 协作三层级 ===

L1 两段式 Prompt
  场景: 单个函数/快速原型
  AI自由度: 高（自由发挥）
  示例: 帮我写一个读取 CSV 的函数

L2 Rules 约束
  场景: 代码风格统一
  AI自由度: 中（风格受限）
  示例: .codebuddy/rules/python-style.md

L3 Spec-driven
  场景: 多模块工程
  AI自由度: 低（行为受限）
  示例: speckit.md 定义接口+约束+验收标准

进阶路径：第1讲 L1 → 第2讲 L1+L2 → 第3讲 L3


### 2.2 speckit.md 完整示例

In [9]:
speckit_example = (
    "# speckit.md -- 科研数据分析工具 Reporter 模块\n\n"
    "## Constitution（硬约束）\n\n"
    "1. 所有公开函数必须有类型注解和 Google style docstring\n"
    "2. 不修改原始输入数据（必须 .copy() 或只读操作）\n"
    "3. 不允许 bare except，必须捕获具体异常\n"
    "4. 文件操作用 pathlib.Path\n"
    "5. 数值结果保留 4 位小数\n"
    "6. 中文注释，英文变量名 snake_case\n"
    "7. 测试覆盖：每个公开函数至少 2 个测试\n\n"
    "## Context（环境说明）\n\n"
    "- Python 3.12, pandas 2.x, pathlib\n"
    "- 已有模块：loader.py, cleaner.py, stats_engine.py, visualizer.py\n"
    "- 数据来源：清洗后的 DataFrame + 统计结果字典\n\n"
    "## Task: reporter.py 模块\n\n"
    "### 接口定义\n"
    "def generate_report(df, stats_result, output_path) -> Path\n\n"
    "### 报告结构\n"
    "1. 数据概览（shape, columns, dtypes 摘要）\n"
    "2. 清洗记录（缺失值/异常值处理统计）\n"
    "3. 统计结果（描述统计/相关性/检验结果）\n"
    "4. 图表引用（引用 visualizer 生成的图片路径）\n\n"
    "### 验收标准\n"
    "- 输出 .md 文件，Markdown 语法正确\n"
    "- 报告 >= 4 个段落\n"
    "- 有完整类型注解\n"
    "- pytest 测试通过"
)

print(speckit_example)

# speckit.md -- 科研数据分析工具 Reporter 模块

## Constitution（硬约束）

1. 所有公开函数必须有类型注解和 Google style docstring
2. 不修改原始输入数据（必须 .copy() 或只读操作）
3. 不允许 bare except，必须捕获具体异常
4. 文件操作用 pathlib.Path
5. 数值结果保留 4 位小数
6. 中文注释，英文变量名 snake_case
7. 测试覆盖：每个公开函数至少 2 个测试

## Context（环境说明）

- Python 3.12, pandas 2.x, pathlib
- 已有模块：loader.py, cleaner.py, stats_engine.py, visualizer.py
- 数据来源：清洗后的 DataFrame + 统计结果字典

## Task: reporter.py 模块

### 接口定义
def generate_report(df, stats_result, output_path) -> Path

### 报告结构
1. 数据概览（shape, columns, dtypes 摘要）
2. 清洗记录（缺失值/异常值处理统计）
3. 统计结果（描述统计/相关性/检验结果）
4. 图表引用（引用 visualizer 生成的图片路径）

### 验收标准
- 输出 .md 文件，Markdown 语法正确
- 报告 >= 4 个段落
- 有完整类型注解
- pytest 测试通过


### 2.3 Spec-driven 完整流程演示

In [10]:
# 演示：从 Spec 到实现的完整循环

print("=== Spec-driven 五步工作流 ===")
print()
steps = [
    ("1. Constitution", "定义硬约束（不可违反的规则）"),
    ("2. Context", "描述环境（依赖/数据/平台）"),
    ("3. Tasks", "定义任务（接口/行为/验收标准）"),
    ("4. AI 实现", "提交 @speckit.md → AI 在约束内编码"),
    ("5. pytest 验收", "测试通过=验收成功；失败=Round2告知错误"),
]
for step, desc in steps:
    print(f"  {step}: {desc}")

print()
print("=== 与 Prompt 的本质区别 ===")
print("  Prompt: '帮我写一个报告生成函数'")
print("  → AI 自由发挥，可能遗漏类型注解/错误处理/...")
print()
print("  Spec:   @speckit.md '按规约实现 reporter.py'")
print("  → AI 在边界内实现，Constitution 规则条条生效")

=== Spec-driven 五步工作流 ===

  1. Constitution: 定义硬约束（不可违反的规则）
  2. Context: 描述环境（依赖/数据/平台）
  3. Tasks: 定义任务（接口/行为/验收标准）
  4. AI 实现: 提交 @speckit.md → AI 在约束内编码
  5. pytest 验收: 测试通过=验收成功；失败=Round2告知错误

=== 与 Prompt 的本质区别 ===
  Prompt: '帮我写一个报告生成函数'
  → AI 自由发挥，可能遗漏类型注解/错误处理/...

  Spec:   @speckit.md '按规约实现 reporter.py'
  → AI 在边界内实现，Constitution 规则条条生效


## Part 3：小结

**受众**：全体学生

**前置条件**：Part 1-2 完成

**学习目标**：回顾三讲的工具线演进

### 知识点速查表

In [11]:
ref = {
    "包结构": "__init__.py + 模块文件",
    "绝对导入": "from sci_analyzer.loader import load_data",
    "类型注解": "def f(x: int, y: Optional[str] = None) -> bool:",
    "自定义异常": "class MyError(Exception): ...",
    "try/except": "try: ... except SpecificError: ... finally: ...",
    "上下文管理器": "with open(f) as fp: ...",
    "pytest 断言": "assert result == expected",
    "fixture": "@pytest.fixture\ndef data(): return ...",
    "参数化": "@pytest.mark.parametrize('x,y', [(1,2), (3,4)])",
    "ruff": "ruff check . && ruff format .",
    "Spec-driven": "constitution + context + tasks → AI实现 → 验收",
}

print("=" * 55)
print("第3讲 知识点速查表")
print("=" * 55)
for k, v in ref.items():
    print(f"  {k:12s} → {v}")

第3讲 知识点速查表
  包结构          → __init__.py + 模块文件
  绝对导入         → from sci_analyzer.loader import load_data
  类型注解         → def f(x: int, y: Optional[str] = None) -> bool:
  自定义异常        → class MyError(Exception): ...
  try/except   → try: ... except SpecificError: ... finally: ...
  上下文管理器       → with open(f) as fp: ...
  pytest 断言    → assert result == expected
  fixture      → @pytest.fixture
def data(): return ...
  参数化          → @pytest.mark.parametrize('x,y', [(1,2), (3,4)])
  ruff         → ruff check . && ruff format .
  Spec-driven  → constitution + context + tasks → AI实现 → 验收


### Vibe Coding 工具速查 + 三讲角色演进

In [12]:
print("=" * 55)
print("三讲 AI 角色演进")
print("=" * 55)
roles = [
    ("第1讲", "编程搭档", "Prompt→生成→审查"),
    ("第2讲", "设计顾问", "方案推荐→对比→选择"),
    ("第3讲", "受控开发者", "Spec约束→AI实现→验收"),
]
for lecture, role, mode in roles:
    print(f"  {lecture}: {role:6s} | {mode}")

print()
print("趋势：AI 自由度递减 → 你的控制力递增 → 系统可靠性递增")
print()
print("下讲预告：第4讲 NLP 基础与 Prompt 工程")
print("  → 进入大模型世界，AI 角色 = Prompt 设计师")

三讲 AI 角色演进
  第1讲: 编程搭档   | Prompt→生成→审查
  第2讲: 设计顾问   | 方案推荐→对比→选择
  第3讲: 受控开发者  | Spec约束→AI实现→验收

趋势：AI 自由度递减 → 你的控制力递增 → 系统可靠性递增

下讲预告：第4讲 NLP 基础与 Prompt 工程
  → 进入大模型世界，AI 角色 = Prompt 设计师
